# Stage 3 - Retrieve

## Data
Chroma DB

## Install
```

```

In [7]:
'''
1. Vectors DB Client
'''
import chromadb

client = chromadb.PersistentClient(path="chroma_kb") 
col = client.get_collection(name="knowledge_base")         

In [9]:
from sentence_transformers import SentenceTransformer
'''
Test 1 search
Correct doc: #140
'''
QUERY = "боюсь, что ребёнок случайно купит фильм с моей карты"
e5 = SentenceTransformer("intfloat/multilingual-e5-base")
q_vec = e5.encode(["query: " + QUERY], normalize_embeddings=True)[0]

res = col.query(
  query_embeddings=[q_vec.tolist()],
  n_results=10,
)           

print(f"\nЗапрос: {QUERY}")
for doc, m, dist in zip(res["documents"][0], 
                        res["metadatas"][0],
                        res["distances"][0]):
  print(f"  dist={dist:.3f} [id: {m['id']}] [doc_id: {m['doc_id']}] [title: {m['title']}] {doc}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 19139.78it/s]



Запрос: боюсь, что ребёнок случайно купит фильм с моей карты
  dist=0.162 [id: 123] [doc_id: 108] [title: Родительский контроль: обзор] Родительский контроль — общее название для настроек, которые ограничивают то, что ребёнок видит и делает в онлайн-кинотеатре. Почти везде он складывается из двух частей: фильтра по возрастным рейтингам и ограничения платных действий. Возрастной рейтинг присваивается фильму при выдаче прокатного удостоверения, сервис только показывает его в карточке и использует как фильтр. Рейтинг относится к фильму целиком, поэтому добрый на вид мультфильм с одной тревожной сценой может стоять на ступень выше, чем ожидают родители. Ограничение платных действий закрывает вторую типичную неприятность — оплату случайным нажатием. На телевизоре это особенно легко: карточка новинки и кнопка с ценой находятся в двух движениях пульта. Сервисы решают это по-разному: одни убирают раздел с покупками из детского режима, другие просят подтверждение перед оплатой.
  dist=0.169 [i

In [10]:
'''
Retrieve - dict
'''
from sentence_transformers import SentenceTransformer

THRESHOLD = 0.83

embedder = SentenceTransformer("intfloat/multilingual-e5-base", device="mps")

def retrieve_dict(query, col=col, embedder=embedder, k=5, threshold=THRESHOLD):
  """Retrieval-пайплайн: embed query -> search -> filter -> return."""
  q_vec = embedder.encode("query: " + query, normalize_embeddings=True)
  res = col.query(query_embeddings=[q_vec.tolist()], n_results=k)
  out = []
  for doc, m, dist in zip(res["documents"][0],
                          res["metadatas"][0],
                          res["distances"][0]):
    sim = 1.0 - dist                        # Косинусное сходство
    if sim >= threshold:                    # Порог по сходству
      out.append({
        'id': m['id'],
        'doc_id': m['doc_id'], 
        'similarity': round(sim, 4), 
        'text': doc, 
      }
    )
  return out

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 26346.80it/s]


In [11]:
'''
Test: all queries (not gap)
'''
import json

# load queries
with open("1_queries.json", encoding="utf-8") as f:
  data = json.load(f)

# filter GAPs
queries = [u for u in data if u.get("kind") != "gap"]


for qi, q in enumerate(queries[:5]):
  ret = retrieve_dict(q["query"])
  print(f"\n[{qi}] {q['query']} {q['relevant_docs']}")
  if not ret:
    print("    (ничего выше порога — пайплайн вернул пусто)")
  for i in ret:
    print(f"    sim={i['similarity']:.3f} #{i['doc_id']} {doc[:58]} #{id}")


[0] что означает ошибка P-102 [85]
    sim=0.861 #85 Вопросы о деньгах разбирает та же поддержка, что и остальн #<built-in function id>
    sim=0.861 #178 Вопросы о деньгах разбирает та же поддержка, что и остальн #<built-in function id>
    sim=0.858 #137 Вопросы о деньгах разбирает та же поддержка, что и остальн #<built-in function id>
    sim=0.856 #138 Вопросы о деньгах разбирает та же поддержка, что и остальн #<built-in function id>
    sim=0.856 #129 Вопросы о деньгах разбирает та же поддержка, что и остальн #<built-in function id>

[1] какая версия webOS или Tizen нужна для приложения на телевизоре [146]
    sim=0.871 #146 Вопросы о деньгах разбирает та же поддержка, что и остальн #<built-in function id>
    sim=0.862 #86 Вопросы о деньгах разбирает та же поддержка, что и остальн #<built-in function id>
    sim=0.838 #129 Вопросы о деньгах разбирает та же поддержка, что и остальн #<built-in function id>
    sim=0.836 #172 Вопросы о деньгах разбирает та же поддержка, что и остал

In [13]:
'''
4.2 Test: compare with relevant docs
'''
# precision@K, recall@K и MRR

relevant = {qi: set(q["relevant_docs"]) for qi, q in enumerate(queries)}

problem = []
for qi, q in enumerate(queries):
    ret = retrieve_dict(q["query"])
    # hits = retrieve(q["query"], k=5, threshold=0.0)
    top_ids = [i['doc_id'] for i in ret]
    rel = relevant[qi]
    is_problem = (not top_ids) or (top_ids[0] not in rel)
    if is_problem:
        problem.append(qi)

print("Проблемные запросы:", problem)

for qi in problem:
    q = queries[qi]
    rel = relevant[qi]
    ret = retrieve_dict(q["query"], k=5)
    print(f"\n[{qi}] {q['query']}  (релевантные id: {sorted(rel) or 'нет'})")
    for i in ret:
        mark = "+" if i['doc_id'] in rel else " "
        print(f"    {mark} sim={i['similarity']:.3f} #{i['id']} {i['text'][:60]}")

# print
# print("\nЗапрос [8] без порога:",
#   [f"#{c}" for c, id, *_ in retrieve(queries[8]['query'], col, embedder,
#                                        k=3, threshold=0.0)])
# print("Запрос [8] с порогом 0.83:",
#   [f"#{c}" for c, id, *_ in retrieve(queries[8]['query'], col, embedder,
#                                        k=3, threshold=0.83)])

Проблемные запросы: [3, 6, 10, 13, 15, 18, 19, 20, 22, 23, 24, 25, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 41, 43, 44, 46, 47, 48, 50, 51, 53, 54]

[3] как подключить саундбар по HDMI ARC  (релевантные id: [22])
      sim=0.842 #145 Если при воспроизведении звук идет, а изображения нет — экра
      sim=0.834 #79 Объёмный звук дома собирают двумя способами, и выбор между н

[6] сколько стоит тариф Максимум и что в него входит  (релевантные id: [113])
      sim=0.855 #28 # Какой тариф подойдёт именно вам В «Фильмоскопе» три тарифа
    + sim=0.844 #132 # Тарифы «Фильмоскопа» Тариф выбирается при оформлении подпи
      sim=0.831 #37 Почему зрители выбирают «Максимум» «Максимум» — это кино без

[10] сколько профилей можно создать в одном аккаунте  (релевантные id: [63])
      sim=0.893 #72 Сколько профилей помещается в один аккаунт и как их создать,
    + sim=0.884 #73 # Профили в аккаунте Профиль — это отдельное пространство вн
      sim=0.868 #71 Аккаунт и профиль — не одно и то ж

In [8]:
'''
5. LLM for Expansion & HyDE
'''
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8080/v1", api_key="not-needed")
LLM = "qwen3.5"                                       # Имя сервер игнорирует
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}


def generate(messages, max_tokens=160):
    """Один вызов локальной LLM. temperature=0 — для воспроизводимости,
    режим рассуждения Qwen3.5 отключён, поэтому сразу приходит готовый текст."""
    resp = client.chat.completions.create(
        model=LLM, messages=messages, temperature=0.0,
        max_tokens=max_tokens, extra_body=NO_THINK)
    return resp.choices[0].message.content.strip()

In [ ]:
'''
6. Expansion and Ranking
'''
from collections import defaultdict

# def search(query, n=50):
#     """Поиск по тексту: кодирует его как query: и возвращает индексы
#     top-n документов по убыванию косинусного сходства."""
#     ret = retrieve(q["query"], col, embedder, k=5, threshold=0.0)
#     return ret


def expand(query, n=5):
  """Query expansion: n перефразировок запроса официальными словами базы."""
  text = generate([
      {"role": "system", "content": "Ты помощник поиска по базе знаний "
                                    "онлайн-кинотеатра."},
      {"role": "user", "content": f"Переформулируй запрос клиента {n} способами "
        f"официальными словами из базы знаний. Используй полные фразы (не одно "
        f"слово), синонимы и термины. По одной формулировке на строку, без "
        f"нумерации.\n\nЗапрос: {query}"}])
  expanded = [l.strip() for l in text.splitlines() if l.strip()][:n]
  # print('\n>>> expand: ', query, '\n', expanded)
  return expanded


def rrf(ranked_lists, k=60, top_k=5):
  """Reciprocal Rank Fusion: сливаем несколько выдач по рангам."""
  scores = defaultdict(float)
  for lst in ranked_lists:
      for rank, doc in enumerate(lst, start=1):
          scores[doc] += 1.0 / (k + rank)
  ranked = sorted(scores.items(), key=lambda x: -x[1])
  return [doc for doc, _ in ranked[:top_k]]

In [15]:
query = queries[30]['query']
rel = set(queries[30]["relevant_docs"])

# basic search
basic = retrieve(query, k=5)
print('>>> query: ', query)
for i in basic:
  print('>>> basic: ', i)

paras = expand(query, n=10)
for i in paras:   
  print('>>> paras: ', i)

# expanded search
for p in paras:
  found = retrieve(p, k=5) 
  print('>>> found: ', found)

# qs = [basic]
# for item in found:
#   if item not in qs:
#       qs.append(item)
# expanded = rrf(qs)

# for item in expanded:
#   print('>>> expanded: ', expanded[:5])


# mark = lambda i: "+" if i in rel else " "
# print(f"\n[{qi}] {query['query']}")

# for p in paras:
#   print("   ·", p)

# print("  обычный поиск top5:",
#   [f"{mark(i[0])}#{i[3]} {i[1][:32]}" for i in basic[:5]])
# print("  query expansion top5:",
#   [f"{mark(i[0])}#{i[3]} {i[1][:32]}" for i in expanded[:5]])

>>> query:  боюсь, что ребёнок случайно купит фильм с моей карты
>>> basic:  (171, 'Возрастной рейтинг в карточке фильма — не оценка редакции и не жанр, а формальная маркировка, с которой картина вышла в прокат. Сервис её показывает, но не назначает. Присваивают рейтинг при выдаче прокатного удостоверения, и относится он к фильму целиком. Из-за этого мультфильм, который выглядит совсем детским, иногда стоит на ступень выше ожидаемого: хватает одной тревожной сцены или одной реплики, чтобы категория поднялась. Обратное тоже бывает — 0+ не означает «для малышей», под этой маркировкой выходит и взрослое кино, в котором просто нет ничего запрещённого. Иностранные мультфильмы приходят со своей маркировкой, и она не совпадает с российской: при выходе в наш прокат категорию присваивают заново. Полностью на рейтинг полагаться не стоит. Он говорит о том, что в картине нет запрещённого для этого возраста, но ничего не говорит о темпе, громкости и о том, испугается ли ваш ребёнок конкретной сцены

In [10]:
'''
6.1 Test expansion on 3 queries
'''

# for qi in [30]:
for qi in [22, 30, 33]:
  query = queries[qi]
  rel = set(queries[qi]["relevant_docs"])

  # basic search
  basic = retrieve(query['query'], k=5)
  # print('>>> basic: ', basic)

  paras = expand(query['query'])
  # paras.append(query['query'])
  # print('>>> paras: ', paras)

  # expanded search
  found = [retrieve(p, k=5) for p in paras]
  # print('>>> found: ', found)

  qs = [basic]
  for item in found:
    if item not in qs:
        qs.append(item)
  expanded = rrf(qs)

  # for item in expanded:
  #   print('>>> expanded: ', expanded[:5])

  
  mark = lambda i: "+" if i in rel else " "
  print(f"\n[{qi}] {query['query']}")
  
  for p in paras:
    print("   ·", p)

  print("  обычный поиск top5:",
    [f"{mark(i[0])}#{i[3]} {i[1][:32]}" for i in basic[:5]])
  print("  query expansion top5:",
    [f"{mark(i[0])}#{i[3]} {i[1][:32]}" for i in expanded[:5]])


[22] начал в метро с телефона, дома хочу досмотреть на большом экране с той же секунды
   · Запрос на воспроизведение фильма с точки останова с мобильного устройства на домашнем телевизоре.
   · Начало просмотра с мобильного телефона и продолжение на большом экране в домашнем кинотеатре.
   · Переход просмотра с мобильного устройства на домашний телевизор с сохранением текущего времени.
   · Запуск фильма с мобильного телефона и продолжение просмотра на большом экране дома.
   · Воспроизведение контента с точки останова с мобильного устройства на домашнем телевизоре.
  обычный поиск top5: [' #24 # Какой тариф подойдёт именно ва', ' #39 Кино давно смотрят не с одного э', ' #138 Ограничения: **скачивать можно н', ' #101 Офлайн-просмотр — это фильмы и с']
  query expansion top5: [' #24 # Какой тариф подойдёт именно ва', ' #24 # Какой тариф подойдёт именно ва', ' #165 Приложение на телевизоре обновля', ' #39 Кино давно смотрят не с одного э', '+#104 Позиция просмотра сохраняется ав']

[30

In [16]:
import numpy as np

def recall_at_k(found, relevant, k=5):
  """Доля релевантных документов в первых k результатах."""
  hits = set(found[:k]) & relevant
  return len(hits) / len(relevant) if relevant else 0.0

rows = {"Обычный поиск": [], "Query expansion": [], "HyDE": []}

for q in queries:
    query, rel = q["query"], set(q["relevant_docs"])

    base = retrieve_dict(query)
    base_docs = [t['doc_id'] for t in base]
    rows["Обычный поиск"].append(recall_at_k(base_docs, rel))

n = len(queries)
print(f"{'Подход':<18}{'recall@5':>10}")
# for name in rows:
print(f"{"Обычный поиск":<18}{np.mean(rows["Обычный поиск"]):>10.3f}")

Подход              recall@5
Обычный поиск          0.616


In [17]:
def basic_search(query):
  ret = retrieve_dict(query)
  doc_ids = [i['doc_id'] for i in ret]
  return doc_ids

In [18]:
def precision_at_k(found_ids, relevant_ids, k):
    """Доля релевантных среди k выданных чанков."""
    hits = [i for i in found_ids[:k] if i in relevant_ids]
    return len(hits) / k


def recall_at_k(found_ids, relevant_ids, k):
    """Доля релевантных чанков, попавших в top-k."""
    hits = [i for i in found_ids[:k] if i in relevant_ids]
    return len(hits) / len(relevant_ids)


def reciprocal_rank(found_ids, relevant_ids):
    """Обратный ранг первого релевантного чанка; 0, если его нет в выдаче."""
    for rank, i in enumerate(found_ids, start=1):
        if i in relevant_ids:
            return 1.0 / rank
    return 0.0


K = 5
rows = []
for q in queries:
    rel = set(q["relevant_docs"])
    found = basic_search(q["query"])
    rows.append({
        "n_rel": len(rel),
        "p": precision_at_k(found, rel, K),
        "r": recall_at_k(found, rel, K),
        "rr": reciprocal_rank(found, rel),
    })

print('>>> Basic Search')
print(f"{'запрос':<44}{'|R|':>4}{'P@5':>7}{'R@5':>7}{'RR':>7}")
for row, q in zip(rows, queries):
    print(f"{q['query'][:42]:<44}{row['n_rel']:>4}"
          f"{row['p']:>7.2f}{row['r']:>7.2f}{row['rr']:>7.2f}")

print(f"\nprecision@5 = {np.mean([r['p'] for r in rows]):.3f}")
print(f"recall@5    = {np.mean([r['r'] for r in rows]):.3f}")
print(f"MRR         = {np.mean([r['rr'] for r in rows]):.3f}")

>>> Basic Search
запрос                                       |R|    P@5    R@5     RR
что означает ошибка P-102                      1   0.20   1.00   1.00
какая версия webOS или Tizen нужна для при     1   0.20   1.00   1.00
обязателен ли HDCP 2.2 для просмотра в 4K      1   0.40   2.00   1.00
как подключить саундбар по HDMI ARC            1   0.00   0.00   0.00
что означает ошибка P-109                      1   0.20   1.00   1.00
что значит ошибка P-111                        1   0.20   1.00   1.00
сколько стоит тариф Максимум и что в него      1   0.20   1.00   0.50
сколько дней длится пробный период             1   0.40   2.00   1.00
какая скорость интернета нужна для 4K          1   0.20   1.00   1.00
есть ли приложение Фильмоскопа для Nintend     1   0.20   1.00   1.00
сколько профилей можно создать в одном акк     1   0.40   2.00   0.50
на каком тарифе доступен звук 5.1              2   0.40   1.00   1.00
сколько устройств можно привязать к аккаун     1   0.20   1.00   1.00
ско

In [23]:
for K in [1, 3, 5, 10, 20]:
  rows = []
  for q in queries:
    rel = set(q["relevant_docs"])
    found = basic_search(q["query"])
    rows.append({"r": recall_at_k(found, rel, K)})
  print(f"K = {K} | recall@5 = {np.mean([r['r'] for r in rows]):.3f}")

K = 1 | recall@5 = 0.330
K = 3 | recall@5 = 0.723
K = 5 | recall@5 = 0.759
K = 10 | recall@5 = 0.759
K = 20 | recall@5 = 0.759


# Stage 3 - Report
- transformation is needed: Query expansion

K = 5